# VCU IMU Analysis

Loads one decoded `LOG####_imu.csv`, subtracts the per-axis accelerometer mean, applies a constant acceleration drift correction that enforces the configured final-velocity endpoint condition, subtracts a per-axis gyro baseline from the first 10 seconds, integrates the adjusted run into velocity and position, then builds a second integration dataframe over a selected start/end time interval.

## Imports

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## Choose Log

In [2]:
# Directory containing decoded LOG####_imu.csv files.
# This works when the notebook is opened from either the repo root or the VCU folder.
DATA_DIR = Path("June11thTestingDataLog_AfterChargingBattery")

# Accepts 10, "10", "0010", or "LOG0010".
LOG_NUMBER = "LOG0010"

# Gyro bias is estimated from the first N seconds of the run.
GYRO_BASELINE_SECONDS = 10.0

# Integration interval in elapsed seconds. Set INTEGRATION_END_S = None to use the final IMU sample.
# This controls the second integration dataframe. The original full-run integration is still calculated first.
# At INTEGRATION_START_S, integrated position and velocity states start at zero.
# The selected interval can optionally apply its own endpoint velocity correction below.
INTEGRATION_START_S = 0.0
INTEGRATION_END_S = None

# The car is assumed to be stationary at the end of the full run. When enabled, subtract a
# constant per-axis acceleration correction so trapezoidal integration ends at this target velocity.
ENFORCE_FULL_RUN_ENDPOINT_VELOCITY = True
FULL_RUN_ENDPOINT_VELOCITY_TARGET_MPS = np.array([0.0, 0.0, 0.0], dtype=float)

# Apply an endpoint velocity condition to the selected interval integration dataframe too.
# Disable this when the selected interval's final sample is not a known stop.
ENFORCE_SELECTED_INTERVAL_ENDPOINT_VELOCITY = True
SELECTED_INTERVAL_ENDPOINT_VELOCITY_TARGET_MPS = np.array([0.0, 0.0, 0.0], dtype=float)

# Use this window to summarize any remaining velocity drift near the end of the full run.
END_REST_WINDOW_SECONDS = 10.0

G_TO_MPS2 = 9.80665
MPS_TO_MPH = 2.2369362920544

## Load IMU Data

In [3]:
ACCEL_COLS = ["accel_x_g", "accel_y_g", "accel_z_g"]
GYRO_COLS = ["gyro_x_dps", "gyro_y_dps", "gyro_z_dps"]
AXES = ["x", "y", "z"]
REQUIRED_COLS = ["t_ms", *ACCEL_COLS, *GYRO_COLS]
LOG_ID_RE = re.compile(r"^LOG(?P<num>\d+)$", re.IGNORECASE)


def normalize_log_id(log_number):
    text = str(log_number).strip()
    match = LOG_ID_RE.match(text)
    if match:
        return f"LOG{int(match.group('num')):04d}"
    if text.isdigit():
        return f"LOG{int(text):04d}"
    raise ValueError(f"Could not parse LOG number from {log_number!r}")


def resolve_data_dir(data_dir):
    data_dir = Path(data_dir).expanduser()
    candidates = [data_dir, Path.cwd() / data_dir, Path.cwd() / "VCU" / data_dir]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find data directory: {data_dir}")


def load_imu_log(data_dir, log_number):
    log_id = normalize_log_id(log_number)
    data_dir = resolve_data_dir(data_dir)
    path = data_dir / f"{log_id}_imu.csv"
    if not path.exists():
        raise FileNotFoundError(f"Could not find IMU log: {path}")

    raw = pd.read_csv(path)
    missing = [column for column in REQUIRED_COLS if column not in raw.columns]
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    imu = raw.copy()
    for column in [*REQUIRED_COLS, "imu_temp_c"]:
        if column in imu.columns:
            imu[column] = pd.to_numeric(imu[column], errors="coerce")

    imu = imu.dropna(subset=REQUIRED_COLS).sort_values("t_ms").drop_duplicates("t_ms")
    imu = imu.reset_index(drop=True)
    if imu.empty:
        raise ValueError(f"{path.name} has no valid IMU samples after numeric cleanup")

    imu.insert(1, "elapsed_s", (imu["t_ms"] - imu["t_ms"].iloc[0]) / 1000.0)
    return log_id, path, raw, imu


def cumulative_trapezoid(values, elapsed_s):
    values = np.asarray(values, dtype=float)
    elapsed_s = np.asarray(elapsed_s, dtype=float)
    squeeze = values.ndim == 1
    if squeeze:
        values = values.reshape(-1, 1)
    if len(values) != len(elapsed_s):
        raise ValueError("values and elapsed_s must have the same number of samples")
    if len(values) == 0:
        return values.ravel() if squeeze else values

    dt_s = np.diff(elapsed_s)
    if np.any(dt_s < 0):
        raise ValueError("elapsed_s must be sorted before integration")

    segment_area = 0.5 * (values[1:] + values[:-1]) * dt_s[:, None]
    integrated = np.vstack([np.zeros((1, values.shape[1])), np.cumsum(segment_area, axis=0)])
    return integrated.ravel() if squeeze else integrated


def validate_velocity_target(target_final_velocity_mps, axis_count):
    target = np.asarray(target_final_velocity_mps, dtype=float)
    if target.shape != (axis_count,):
        raise ValueError(f"Endpoint velocity target must have shape ({axis_count},), got {target.shape}")
    return target


def apply_endpoint_velocity_constraint(accel_mps2, elapsed_s, target_final_velocity_mps):
    accel_mps2 = np.asarray(accel_mps2, dtype=float)
    elapsed_s = np.asarray(elapsed_s, dtype=float)
    if accel_mps2.ndim != 2:
        raise ValueError("accel_mps2 must be a 2D array")
    if len(accel_mps2) < 2:
        raise ValueError("Endpoint velocity constraint requires at least two samples")

    target_final_velocity_mps = validate_velocity_target(target_final_velocity_mps, accel_mps2.shape[1])
    duration_s = float(elapsed_s[-1] - elapsed_s[0])
    if duration_s <= 0.0:
        raise ValueError("Endpoint velocity constraint requires a positive duration")

    unconstrained_velocity_mps = cumulative_trapezoid(accel_mps2, elapsed_s)
    pre_constraint_final_velocity_mps = unconstrained_velocity_mps[-1]
    accel_correction_mps2 = (pre_constraint_final_velocity_mps - target_final_velocity_mps) / duration_s
    constrained_accel_mps2 = accel_mps2 - accel_correction_mps2
    constrained_velocity_mps = cumulative_trapezoid(constrained_accel_mps2, elapsed_s)
    return (
        constrained_accel_mps2,
        constrained_velocity_mps,
        accel_correction_mps2,
        pre_constraint_final_velocity_mps,
    )


log_id, imu_path, raw_imu, imu = load_imu_log(DATA_DIR, LOG_NUMBER)

print(f"Loaded {imu_path}")
print(f"Rows loaded: {len(raw_imu):,}")
print(f"Rows after numeric cleanup: {len(imu):,}")
print(f"Elapsed time: {imu['elapsed_s'].iloc[0]:.3f} s to {imu['elapsed_s'].iloc[-1]:.3f} s")
display(imu.head())

Loaded C:\Users\Nathan\Documents\Formula_E\BERT2026\BERT26-Firmware\BERT26-Firmware\VCU\June11thTestingDataLog_AfterChargingBattery\LOG0010_imu.csv
Rows loaded: 18,897
Rows after numeric cleanup: 18,897
Elapsed time: 0.000 s to 199.095 s


,t_ms,elapsed_s,accel_x_g,accel_y_g,accel_z_g,gyro_x_dps,gyro_y_dps,gyro_z_dps,imu_temp_c
0,3280,0.00,0.2004,-0.0433,0.8828,-6.11,8.05,0.02,23.00
1,3290,0.01,0.2136,-0.0348,0.8798,-6.03,7.33,-0.03,23.00
2,3300,0.02,0.2021,-0.0403,0.8848,-6.08,6.75,0.00,23.02
3,3310,0.03,0.2231,-0.0409,0.9000,-6.14,8.70,-0.02,23.02
4,3320,0.04,0.2224,-0.0375,0.8947,-6.17,7.18,-0.09,23.02


## Normalize Acceleration And Calibrate Gyro

In [4]:
accel_mean_g = imu[ACCEL_COLS].mean()

gyro_baseline_mask = imu["elapsed_s"] <= GYRO_BASELINE_SECONDS
if not gyro_baseline_mask.any():
    raise ValueError(f"No samples found in the first {GYRO_BASELINE_SECONDS} seconds")
gyro_baseline_dps = imu.loc[gyro_baseline_mask, GYRO_COLS].mean()

imu_adjusted = imu.copy()
accel_adj_g_cols = []
accel_adj_mps2_cols = []
gyro_adj_dps_cols = []
gyro_adj_radps_cols = []

for axis, source_col in zip(AXES, ACCEL_COLS):
    adj_g_col = f"accel_{axis}_adj_g"
    adj_mps2_col = f"accel_{axis}_adj_mps2"
    imu_adjusted[adj_g_col] = imu_adjusted[source_col] - accel_mean_g[source_col]
    imu_adjusted[adj_mps2_col] = imu_adjusted[adj_g_col] * G_TO_MPS2
    accel_adj_g_cols.append(adj_g_col)
    accel_adj_mps2_cols.append(adj_mps2_col)

for axis, source_col in zip(AXES, GYRO_COLS):
    adj_dps_col = f"gyro_{axis}_adj_dps"
    adj_radps_col = f"gyro_{axis}_adj_radps"
    imu_adjusted[adj_dps_col] = imu_adjusted[source_col] - gyro_baseline_dps[source_col]
    imu_adjusted[adj_radps_col] = np.deg2rad(imu_adjusted[adj_dps_col])
    gyro_adj_dps_cols.append(adj_dps_col)
    gyro_adj_radps_cols.append(adj_radps_col)

calibration_summary = pd.DataFrame(
    {
        "axis": AXES,
        "accel_mean_g_subtracted": [accel_mean_g[col] for col in ACCEL_COLS],
        "gyro_first_10s_mean_dps_subtracted": [gyro_baseline_dps[col] for col in GYRO_COLS],
    }
)

print(f"Gyro baseline samples: {int(gyro_baseline_mask.sum()):,}")
display(calibration_summary)
display(imu_adjusted[["elapsed_s", *ACCEL_COLS, *accel_adj_g_cols, *GYRO_COLS, *gyro_adj_dps_cols]].head())

Gyro baseline samples: 997


,axis,accel_mean_g_subtracted,gyro_first_10s_mean_dps_subtracted
0,x,0.201356,-6.085607
1,y,-0.087372,7.766459
2,z,0.887690,-0.046580


,elapsed_s,accel_x_g,accel_y_g,accel_z_g,accel_x_adj_g,accel_y_adj_g,accel_z_adj_g,gyro_x_dps,gyro_y_dps,gyro_z_dps,gyro_x_adj_dps,gyro_y_adj_dps,gyro_z_adj_dps
0,0.00,0.2004,-0.0433,0.8828,-0.000956,0.044072,-0.00489,-6.11,8.05,0.02,-0.024393,0.283541,0.06658
1,0.01,0.2136,-0.0348,0.8798,0.012244,0.052572,-0.00789,-6.03,7.33,-0.03,0.055607,-0.436459,0.01658
2,0.02,0.2021,-0.0403,0.8848,0.000744,0.047072,-0.00289,-6.08,6.75,0.00,0.005607,-1.016459,0.04658
3,0.03,0.2231,-0.0409,0.9000,0.021744,0.046472,0.01231,-6.14,8.70,-0.02,-0.054393,0.933541,0.02658
4,0.04,0.2224,-0.0375,0.8947,0.021044,0.049872,0.00701,-6.17,7.18,-0.09,-0.084393,-0.586459,-0.04342


## Integrate Adjusted Full Run

In [5]:
elapsed_s = imu_adjusted["elapsed_s"].to_numpy(dtype=float)
base_accel_adj_mps2 = np.column_stack(
    [
        (imu_adjusted[source_col].to_numpy(dtype=float) - accel_mean_g[source_col]) * G_TO_MPS2
        for source_col in ACCEL_COLS
    ]
)
gyro_adj_dps = imu_adjusted[gyro_adj_dps_cols].to_numpy(dtype=float)

endpoint_velocity_target_mps = validate_velocity_target(
    FULL_RUN_ENDPOINT_VELOCITY_TARGET_MPS,
    len(AXES),
)

if ENFORCE_FULL_RUN_ENDPOINT_VELOCITY:
    (
        accel_adj_mps2,
        velocity_mps,
        endpoint_accel_correction_mps2,
        pre_endpoint_final_velocity_mps,
    ) = apply_endpoint_velocity_constraint(
        base_accel_adj_mps2,
        elapsed_s,
        endpoint_velocity_target_mps,
    )
else:
    accel_adj_mps2 = base_accel_adj_mps2
    velocity_mps = cumulative_trapezoid(accel_adj_mps2, elapsed_s)
    endpoint_accel_correction_mps2 = np.zeros(len(AXES), dtype=float)
    pre_endpoint_final_velocity_mps = velocity_mps[-1]

velocity_cols = [f"vel_{axis}_mps" for axis in AXES]
position_cols = [f"pos_{axis}_m" for axis in AXES]
angle_cols = [f"angle_{axis}_deg" for axis in AXES]

position_m = cumulative_trapezoid(velocity_mps, elapsed_s)
angle_deg = cumulative_trapezoid(gyro_adj_dps, elapsed_s)

for col_index, (adj_g_col, adj_mps2_col) in enumerate(zip(accel_adj_g_cols, accel_adj_mps2_cols)):
    imu_adjusted[adj_mps2_col] = accel_adj_mps2[:, col_index]
    imu_adjusted[adj_g_col] = accel_adj_mps2[:, col_index] / G_TO_MPS2

imu_adjusted[velocity_cols] = velocity_mps
imu_adjusted[position_cols] = position_m
imu_adjusted[angle_cols] = angle_deg

post_endpoint_final_velocity_mps = velocity_mps[-1]
endpoint_constraint_table = pd.DataFrame(
    {
        "axis": AXES,
        "target_final_velocity_mps": endpoint_velocity_target_mps,
        "pre_constraint_final_velocity_mps": pre_endpoint_final_velocity_mps,
        "accel_correction_mps2": endpoint_accel_correction_mps2,
        "accel_correction_g": endpoint_accel_correction_mps2 / G_TO_MPS2,
        "post_constraint_final_velocity_mps": post_endpoint_final_velocity_mps,
        "post_constraint_residual_mps": post_endpoint_final_velocity_mps - endpoint_velocity_target_mps,
    }
)

if ENFORCE_FULL_RUN_ENDPOINT_VELOCITY:
    print("Applied full-run endpoint velocity constraint before integration.")
else:
    print("Full-run endpoint velocity constraint disabled.")
print("Integrated the full adjusted run into imu_adjusted velocity and position columns.")
print("Integrated calibrated angular velocity into angle columns.")
display(endpoint_constraint_table)
display(imu_adjusted[["elapsed_s", *accel_adj_mps2_cols, *velocity_cols, *position_cols, *angle_cols]].tail())

Applied full-run endpoint velocity constraint before integration.
Integrated the full adjusted run into imu_adjusted velocity and position columns.
Integrated calibrated angular velocity into angle columns.


,axis,target_final_velocity_mps,pre_constraint_final_velocity_mps,accel_correction_mps2,accel_correction_g,post_constraint_final_velocity_mps,post_constraint_residual_mps
0,x,0.0,-5.418851,-0.027217,-0.002775,-1.107003e-14,-1.107003e-14
1,y,0.0,-14.134954,-0.070996,-0.007240,-1.726822e-13,-1.726822e-13
2,z,0.0,-0.225495,-0.001133,-0.000115,3.938039e-15,3.938039e-15


,elapsed_s,accel_x_adj_mps2,accel_y_adj_mps2,accel_z_adj_mps2,vel_x_mps,vel_y_mps,vel_z_mps,pos_x_m,pos_y_m,pos_z_m,angle_x_deg,angle_y_deg,angle_z_deg
18892,199.055,-0.005697,0.758168,-0.037011,3.871053e-03,-2.961083e-02,-3.926137e-04,421.757587,-3083.878831,-276.919239,42.780196,-2.468497,517.192270
18893,199.065,0.030588,0.749342,-0.035050,3.995505e-03,-2.207328e-02,-7.529212e-04,421.757626,-3083.879090,-276.919245,42.778152,-2.471961,517.192086
18894,199.075,-0.262631,0.700309,0.136566,2.835287e-03,-1.482503e-02,-2.453401e-04,421.757661,-3083.879274,-276.919250,42.776008,-2.477626,517.192002
18895,199.085,-0.139068,0.743458,0.005157,8.267926e-04,-7.606196e-03,4.632773e-04,421.757679,-3083.879387,-276.919249,42.773564,-2.471240,517.191867
18896,199.095,-0.026291,0.777781,-0.097813,-1.107003e-14,-1.726822e-13,3.938039e-15,421.757683,-3083.879425,-276.919247,42.771220,-2.467405,517.192033


## Full-Run Endpoint Velocity Check

In [6]:
end_window_start_s = max(elapsed_s[-1] - END_REST_WINDOW_SECONDS, elapsed_s[0])
end_rest_mask = imu_adjusted["elapsed_s"] >= end_window_start_s

final_velocity_mps = imu_adjusted.loc[imu_adjusted.index[-1], velocity_cols].to_numpy(dtype=float)
final_velocity_error_mps = final_velocity_mps - endpoint_velocity_target_mps
end_rest_mean_velocity_mps = imu_adjusted.loc[end_rest_mask, velocity_cols].mean().to_numpy(dtype=float)
end_rest_mean_velocity_error_mps = end_rest_mean_velocity_mps - endpoint_velocity_target_mps

velocity_error_table = pd.DataFrame(
    {
        "axis": AXES,
        "target_final_mps": endpoint_velocity_target_mps,
        "final_sample_mps": final_velocity_mps,
        "final_residual_mps": final_velocity_error_mps,
        "final_residual_mph": final_velocity_error_mps * MPS_TO_MPH,
        "end_rest_mean_mps": end_rest_mean_velocity_mps,
        "end_rest_mean_residual_mps": end_rest_mean_velocity_error_mps,
        "end_rest_mean_residual_mph": end_rest_mean_velocity_error_mps * MPS_TO_MPH,
    }
)
velocity_error_table = pd.concat(
    [
        velocity_error_table,
        pd.DataFrame(
            [
                {
                    "axis": "magnitude",
                    "target_final_mps": np.linalg.norm(endpoint_velocity_target_mps),
                    "final_sample_mps": np.linalg.norm(final_velocity_mps),
                    "final_residual_mps": np.linalg.norm(final_velocity_error_mps),
                    "final_residual_mph": np.linalg.norm(final_velocity_error_mps) * MPS_TO_MPH,
                    "end_rest_mean_mps": np.linalg.norm(end_rest_mean_velocity_mps),
                    "end_rest_mean_residual_mps": np.linalg.norm(end_rest_mean_velocity_error_mps),
                    "end_rest_mean_residual_mph": np.linalg.norm(end_rest_mean_velocity_error_mps) * MPS_TO_MPH,
                }
            ]
        ),
    ],
    ignore_index=True,
)

print(f"Assuming the car is at rest from {end_window_start_s:.3f} s to {elapsed_s[-1]:.3f} s.")
print(
    "Final velocity residual magnitude: "
    f"{np.linalg.norm(final_velocity_error_mps):.6f} m/s "
    f"({np.linalg.norm(final_velocity_error_mps) * MPS_TO_MPH:.6f} mph)"
)
print(
    "Mean velocity residual magnitude over final rest window: "
    f"{np.linalg.norm(end_rest_mean_velocity_error_mps):.6f} m/s "
    f"({np.linalg.norm(end_rest_mean_velocity_error_mps) * MPS_TO_MPH:.6f} mph)"
)
display(velocity_error_table)

Assuming the car is at rest from 189.095 s to 199.095 s.
Final velocity residual magnitude: 0.000000 m/s (0.000000 mph)
Mean velocity residual magnitude over final rest window: 3.746915 m/s (8.381611 mph)


,axis,target_final_mps,final_sample_mps,final_residual_mps,final_residual_mph,end_rest_mean_mps,end_rest_mean_residual_mps,end_rest_mean_residual_mph
0,x,0.0,-1.107003e-14,-1.107003e-14,-2.476295e-14,0.555997,0.555997,1.243730
1,y,0.0,-1.726822e-13,-1.726822e-13,-3.862790e-13,-3.698821,-3.698821,-8.274027
2,z,0.0,3.938039e-15,3.938039e-15,8.809143e-15,-0.221280,-0.221280,-0.494990
3,magnitude,0.0,1.730815e-13,1.730815e-13,3.871722e-13,3.746915,3.746915,8.381611


## Integration Window Functions

In [7]:
def select_integration_window(df, start_s, end_s=None):
    start_s = float(start_s)
    run_start_s = float(df['elapsed_s'].iloc[0])
    run_end_s = float(df['elapsed_s'].iloc[-1])
    end_s = run_end_s if end_s is None else float(end_s)

    if start_s < run_start_s or start_s > run_end_s:
        raise ValueError(f'INTEGRATION_START_S must be between {run_start_s:.3f} and {run_end_s:.3f} seconds')
    if end_s < start_s:
        raise ValueError('INTEGRATION_END_S must be greater than or equal to INTEGRATION_START_S')
    if end_s > run_end_s:
        raise ValueError(f'INTEGRATION_END_S must be no later than {run_end_s:.3f} seconds')

    mask = (df['elapsed_s'] >= start_s) & (df['elapsed_s'] <= end_s)
    window = df.loc[mask].copy().reset_index(drop=True)
    if len(window) < 2:
        raise ValueError('Integration interval must include at least two IMU samples')

    actual_start_s = float(window['elapsed_s'].iloc[0])
    actual_end_s = float(window['elapsed_s'].iloc[-1])
    window.insert(2, 'integration_elapsed_s', window['elapsed_s'] - actual_start_s)

    window_info = {
        'requested_start_s': start_s,
        'requested_end_s': end_s,
        'actual_start_s': actual_start_s,
        'actual_end_s': actual_end_s,
        'duration_s': actual_end_s - actual_start_s,
        'samples': len(window),
    }
    return window, window_info


def copy_signal_columns(df, source_cols, output_cols):
    for source_col, output_col in zip(source_cols, output_cols):
        df[output_col] = df[source_col]


def build_integration_dataframe(
    source_df,
    start_s,
    end_s=None,
    enforce_endpoint_velocity=False,
    endpoint_velocity_target_mps=None,
):
    integration_df, window_info = select_integration_window(source_df, start_s, end_s)

    accel_integration_g_cols = [f'accel_{axis}_integration_g' for axis in AXES]
    accel_integration_mps2_cols = [f'accel_{axis}_integration_mps2' for axis in AXES]
    gyro_integration_dps_cols = [f'gyro_{axis}_integration_dps' for axis in AXES]
    gyro_integration_radps_cols = [f'gyro_{axis}_integration_radps' for axis in AXES]
    velocity_cols = [f'vel_{axis}_mps' for axis in AXES]
    position_cols = [f'pos_{axis}_m' for axis in AXES]
    angle_cols = [f'angle_{axis}_deg' for axis in AXES]

    copy_signal_columns(integration_df, accel_adj_g_cols, accel_integration_g_cols)
    copy_signal_columns(integration_df, accel_adj_mps2_cols, accel_integration_mps2_cols)
    copy_signal_columns(integration_df, gyro_adj_dps_cols, gyro_integration_dps_cols)

    for dps_col, radps_col in zip(gyro_integration_dps_cols, gyro_integration_radps_cols):
        integration_df[radps_col] = np.deg2rad(integration_df[dps_col])

    local_elapsed_s = integration_df['integration_elapsed_s'].to_numpy(dtype=float)
    accel_mps2 = integration_df[accel_integration_mps2_cols].to_numpy(dtype=float)
    gyro_dps = integration_df[gyro_integration_dps_cols].to_numpy(dtype=float)
    endpoint_velocity_target_mps = (
        np.zeros(len(AXES), dtype=float)
        if endpoint_velocity_target_mps is None
        else validate_velocity_target(endpoint_velocity_target_mps, len(AXES))
    )

    if enforce_endpoint_velocity:
        (
            accel_mps2,
            velocity_mps,
            endpoint_accel_correction_mps2,
            pre_endpoint_final_velocity_mps,
        ) = apply_endpoint_velocity_constraint(
            accel_mps2,
            local_elapsed_s,
            endpoint_velocity_target_mps,
        )
        for col_index, (accel_g_col, accel_mps2_col) in enumerate(
            zip(accel_integration_g_cols, accel_integration_mps2_cols)
        ):
            integration_df[accel_mps2_col] = accel_mps2[:, col_index]
            integration_df[accel_g_col] = accel_mps2[:, col_index] / G_TO_MPS2
    else:
        velocity_mps = cumulative_trapezoid(accel_mps2, local_elapsed_s)
        endpoint_accel_correction_mps2 = np.zeros(len(AXES), dtype=float)
        pre_endpoint_final_velocity_mps = velocity_mps[-1]
    position_m = cumulative_trapezoid(velocity_mps, local_elapsed_s)
    angle_deg = cumulative_trapezoid(gyro_dps, local_elapsed_s)

    integration_df[velocity_cols] = velocity_mps
    integration_df[position_cols] = position_m
    integration_df[angle_cols] = angle_deg

    window_info['initial_position_m'] = 0.0
    window_info['initial_velocity_mps'] = 0.0
    window_info['endpoint_velocity_constraint_enforced'] = bool(enforce_endpoint_velocity)
    window_info['endpoint_velocity_target_mps'] = endpoint_velocity_target_mps
    window_info['pre_endpoint_final_velocity_mps'] = pre_endpoint_final_velocity_mps
    window_info['endpoint_accel_correction_mps2'] = endpoint_accel_correction_mps2
    window_info['endpoint_accel_correction_g'] = endpoint_accel_correction_mps2 / G_TO_MPS2
    window_info['post_endpoint_final_velocity_mps'] = velocity_mps[-1]
    window_info['post_endpoint_velocity_residual_mps'] = velocity_mps[-1] - endpoint_velocity_target_mps

    integration_columns = {
        'accel_g': accel_integration_g_cols,
        'accel_mps2': accel_integration_mps2_cols,
        'gyro_dps': gyro_integration_dps_cols,
        'gyro_radps': gyro_integration_radps_cols,
        'velocity_mps': velocity_cols,
        'position_m': position_cols,
        'angle_deg': angle_cols,
    }
    return integration_df, window_info, integration_columns


## Integrate Selected Interval Into Second Dataframe

In [8]:
second_integration_df, second_integration_window_info, second_integration_columns = build_integration_dataframe(
    imu_adjusted,
    start_s=INTEGRATION_START_S,
    end_s=INTEGRATION_END_S,
    enforce_endpoint_velocity=ENFORCE_SELECTED_INTERVAL_ENDPOINT_VELOCITY,
    endpoint_velocity_target_mps=SELECTED_INTERVAL_ENDPOINT_VELOCITY_TARGET_MPS,
)

second_accel_integration_g_cols = second_integration_columns['accel_g']
second_accel_integration_mps2_cols = second_integration_columns['accel_mps2']
second_gyro_integration_dps_cols = second_integration_columns['gyro_dps']
second_gyro_integration_radps_cols = second_integration_columns['gyro_radps']
second_velocity_cols = second_integration_columns['velocity_mps']
second_position_cols = second_integration_columns['position_m']
second_angle_cols = second_integration_columns['angle_deg']

second_integration_info_display = {
    key: value
    for key, value in second_integration_window_info.items()
    if np.isscalar(value)
}
second_endpoint_constraint_table = pd.DataFrame(
    {
        'axis': AXES,
        'target_final_velocity_mps': second_integration_window_info['endpoint_velocity_target_mps'],
        'pre_constraint_final_velocity_mps': second_integration_window_info['pre_endpoint_final_velocity_mps'],
        'accel_correction_mps2': second_integration_window_info['endpoint_accel_correction_mps2'],
        'accel_correction_g': second_integration_window_info['endpoint_accel_correction_g'],
        'post_constraint_final_velocity_mps': second_integration_window_info['post_endpoint_final_velocity_mps'],
        'post_constraint_residual_mps': second_integration_window_info['post_endpoint_velocity_residual_mps'],
    }
)

print('Integrated selected adjusted IMU interval into second_integration_df.')
if ENFORCE_SELECTED_INTERVAL_ENDPOINT_VELOCITY:
    print('Applied selected-interval endpoint velocity constraint before integration.')
else:
    print('Selected-interval endpoint velocity constraint disabled.')
print(
    f"Requested interval: {second_integration_window_info['requested_start_s']:.3f} s "
    f"to {second_integration_window_info['requested_end_s']:.3f} s"
)
print(
    f"Actual sample interval: {second_integration_window_info['actual_start_s']:.3f} s "
    f"to {second_integration_window_info['actual_end_s']:.3f} s "
    f"({second_integration_window_info['samples']:,} samples)"
)
display(pd.Series(second_integration_info_display, name='second_integration_window'))
display(second_endpoint_constraint_table)
display(second_integration_df[['elapsed_s', 'integration_elapsed_s', *second_accel_integration_mps2_cols, *second_gyro_integration_dps_cols, *second_velocity_cols, *second_position_cols, *second_angle_cols]].head())
display(second_integration_df[['elapsed_s', 'integration_elapsed_s', *second_accel_integration_mps2_cols, *second_gyro_integration_dps_cols, *second_velocity_cols, *second_position_cols, *second_angle_cols]].tail())

Integrated selected adjusted IMU interval into second_integration_df.
Applied selected-interval endpoint velocity constraint before integration.
Requested interval: 0.000 s to 199.095 s
Actual sample interval: 0.000 s to 199.095 s (18,897 samples)


requested_start_s                            0.0
requested_end_s                          199.095
actual_start_s                               0.0
actual_end_s                             199.095
duration_s                               199.095
samples                                    18897
initial_position_m                           0.0
initial_velocity_mps                         0.0
endpoint_velocity_constraint_enforced       True
Name: second_integration_window, dtype: object

,axis,target_final_velocity_mps,pre_constraint_final_velocity_mps,accel_correction_mps2,accel_correction_g,post_constraint_final_velocity_mps,post_constraint_residual_mps
0,x,0.0,-1.107003e-14,-5.560175e-17,-5.669800e-18,-5.573016e-15,-5.573016e-15
1,y,0.0,-1.726822e-13,-8.673356e-16,-8.844362e-17,-4.763551e-15,-4.763551e-15
2,z,0.0,3.938039e-15,1.977970e-17,2.016968e-18,1.394826e-15,1.394826e-15


,elapsed_s,integration_elapsed_s,accel_x_integration_mps2,accel_y_integration_mps2,accel_z_integration_mps2,gyro_x_integration_dps,gyro_y_integration_dps,gyro_z_integration_dps,vel_x_mps,vel_y_mps,vel_z_mps,pos_x_m,pos_y_m,pos_z_m,angle_x_deg,angle_y_deg,angle_z_deg
0,0.00,0.00,0.017839,0.503195,-0.046818,-0.024393,0.283541,0.06658,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.01,0.01,0.147287,0.586552,-0.076238,0.055607,-0.436459,0.01658,0.000826,0.005449,-0.000615,0.000004,0.000027,-0.000003,0.000156,-0.000765,0.000416
2,0.02,0.02,0.034510,0.532615,-0.027205,0.005607,-1.016459,0.04658,0.001735,0.011045,-0.001132,0.000017,0.000110,-0.000012,0.000462,-0.008029,0.000732
3,0.03,0.03,0.240450,0.526731,0.121856,-0.054393,0.933541,0.02658,0.003109,0.016341,-0.000659,0.000041,0.000247,-0.000021,0.000218,-0.008444,0.001097
4,0.04,0.04,0.233585,0.560074,0.069881,-0.084393,-0.586459,-0.04342,0.005480,0.021775,0.000299,0.000084,0.000437,-0.000023,-0.000476,-0.006708,0.001013


,elapsed_s,integration_elapsed_s,accel_x_integration_mps2,accel_y_integration_mps2,accel_z_integration_mps2,gyro_x_integration_dps,gyro_y_integration_dps,gyro_z_integration_dps,vel_x_mps,vel_y_mps,vel_z_mps,pos_x_m,pos_y_m,pos_z_m,angle_x_deg,angle_y_deg,angle_z_deg
18892,199.055,199.055,-0.005697,0.758168,-0.037011,-0.234393,0.523541,-0.03342,3.871053e-03,-2.961083e-02,-3.926137e-04,421.757587,-3083.878831,-276.919239,42.780196,-2.468497,517.192270
18893,199.065,199.065,0.030588,0.749342,-0.035050,-0.174393,-1.216459,-0.00342,3.995505e-03,-2.207328e-02,-7.529212e-04,421.757626,-3083.879090,-276.919245,42.778152,-2.471961,517.192086
18894,199.075,199.075,-0.262631,0.700309,0.136566,-0.254393,0.083541,-0.01342,2.835287e-03,-1.482503e-02,-2.453401e-04,421.757661,-3083.879274,-276.919250,42.776008,-2.477626,517.192002
18895,199.085,199.085,-0.139068,0.743458,0.005157,-0.234393,1.193541,-0.01342,8.267926e-04,-7.606196e-03,4.632773e-04,421.757679,-3083.879387,-276.919249,42.773564,-2.471240,517.191867
18896,199.095,199.095,-0.026291,0.777781,-0.097813,-0.234393,-0.426459,0.04658,-5.573016e-15,-4.763551e-15,1.394826e-15,421.757683,-3083.879425,-276.919247,42.771220,-2.467405,517.192033


## Selected-Interval Final State Values

In [9]:
second_final_sample = second_integration_df.iloc[-1]
second_endpoint_velocity_target_mps = second_integration_window_info['endpoint_velocity_target_mps']

second_final_position_m = second_final_sample[second_position_cols].to_numpy(dtype=float)
second_final_velocity_mps = second_final_sample[second_velocity_cols].to_numpy(dtype=float)
second_final_velocity_residual_mps = second_final_velocity_mps - second_endpoint_velocity_target_mps
second_final_acceleration_mps2 = second_final_sample[second_accel_integration_mps2_cols].to_numpy(dtype=float)
second_final_acceleration_g = second_final_sample[second_accel_integration_g_cols].to_numpy(dtype=float)
second_final_angular_velocity_dps = second_final_sample[second_gyro_integration_dps_cols].to_numpy(dtype=float)

second_final_state_table = pd.DataFrame(
    {
        'axis': AXES,
        'position_m': second_final_position_m,
        'target_final_velocity_mps': second_endpoint_velocity_target_mps,
        'velocity_mps': second_final_velocity_mps,
        'velocity_mph': second_final_velocity_mps * MPS_TO_MPH,
        'velocity_residual_mps': second_final_velocity_residual_mps,
        'velocity_residual_mph': second_final_velocity_residual_mps * MPS_TO_MPH,
        'acceleration_mps2': second_final_acceleration_mps2,
        'acceleration_g': second_final_acceleration_g,
        'angular_velocity_dps': second_final_angular_velocity_dps,
    }
)
second_final_state_table = pd.concat(
    [
        second_final_state_table,
        pd.DataFrame(
            [
                {
                    'axis': 'magnitude',
                    'position_m': np.linalg.norm(second_final_position_m),
                    'target_final_velocity_mps': np.linalg.norm(second_endpoint_velocity_target_mps),
                    'velocity_mps': np.linalg.norm(second_final_velocity_mps),
                    'velocity_mph': np.linalg.norm(second_final_velocity_mps) * MPS_TO_MPH,
                    'velocity_residual_mps': np.linalg.norm(second_final_velocity_residual_mps),
                    'velocity_residual_mph': np.linalg.norm(second_final_velocity_residual_mps) * MPS_TO_MPH,
                    'acceleration_mps2': np.linalg.norm(second_final_acceleration_mps2),
                    'acceleration_g': np.linalg.norm(second_final_acceleration_g),
                    'angular_velocity_dps': np.linalg.norm(second_final_angular_velocity_dps),
                }
            ]
        ),
    ],
    ignore_index=True,
)

print(
    f"Second integration final state at elapsed_s={second_integration_window_info['actual_end_s']:.3f} s "
    f"(integration_elapsed_s={second_final_sample['integration_elapsed_s']:.3f} s)"
)
print(f"Final position magnitude: {np.linalg.norm(second_final_position_m):.6f} m")
print(
    f"Final velocity magnitude: {np.linalg.norm(second_final_velocity_mps):.6f} m/s "
    f"({np.linalg.norm(second_final_velocity_mps) * MPS_TO_MPH:.6f} mph)"
)
print(
    f"Final velocity residual magnitude: {np.linalg.norm(second_final_velocity_residual_mps):.6f} m/s "
    f"({np.linalg.norm(second_final_velocity_residual_mps) * MPS_TO_MPH:.6f} mph)"
)
print(
    f"Final acceleration magnitude: {np.linalg.norm(second_final_acceleration_mps2):.6f} m/s^2 "
    f"({np.linalg.norm(second_final_acceleration_g):.6f} g)"
)
display(second_final_state_table)

Second integration final state at elapsed_s=199.095 s (integration_elapsed_s=199.095 s)
Final position magnitude: 3124.880177 m
Final velocity magnitude: 0.000000 m/s (0.000000 mph)
Final velocity residual magnitude: 0.000000 m/s (0.000000 mph)
Final acceleration magnitude: 0.784348 m/s^2 (0.079981 g)


,axis,position_m,target_final_velocity_mps,velocity_mps,velocity_mph,velocity_residual_mps,velocity_residual_mph,acceleration_mps2,acceleration_g,angular_velocity_dps
0,x,421.757683,0.0,-5.573016e-15,-1.246648e-14,-5.573016e-15,-1.246648e-14,-0.026291,-0.002681,-0.234393
1,y,-3083.879425,0.0,-4.763551e-15,-1.065576e-14,-4.763551e-15,-1.065576e-14,0.777781,0.079312,-0.426459
2,z,-276.919247,0.0,1.394826e-15,3.120137e-15,1.394826e-15,3.120137e-15,-0.097813,-0.009974,0.046580
3,magnitude,3124.880177,0.0,7.462939e-15,1.669412e-14,7.462939e-15,1.669412e-14,0.784348,0.079981,0.488853
